# Revision logs modelo
---

In [1]:
import os
import datetime
from dotenv import load_dotenv
import requests


In [2]:
# Agrego la ruta del proyecto al path para poder importar los módulos
import sys
import os
# Obtenemos la ruta del directorio del proyecto (parent del notebook)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)

In [3]:
# Importamos nuestros propios módulos (asumiendo que están en la carpeta src)
from src.extractor_ccxt import ExtractorDatosCCXT
from src.model_handler import cargar_modelo, predecir_señal
from src.execution import BinanceExecutor

In [4]:
# Cargamos las variables de entorno (API Keys) de forma segura
load_dotenv()

True

## Extraccion de datos para la nueva predicción

In [5]:
try:
    today     = datetime.datetime.now().strftime("%Y-%m-%d")
    yesterday = (datetime.datetime.now() - datetime.timedelta(days=1)).strftime("%Y-%m-%d")

    # 1. EXTRACCIÓN Y FEATURES
    print("[1/4] Descargando datos y calculando indicadores...")
    extractor = ExtractorDatosCCXT(exchange_id='binance', symbol='BTC/USDT', timeframe='1d')
    
    # El extractor debe encargarse del over-fetch internamente y devolver el df listo
    df_mercado = extractor.obtener_datos(start_date=yesterday, end_date=today)
    df_mercado = extractor.agregar_indicadores_avanzados(df_mercado)
    df_mercado = extractor.agregar_contexto_macro(df_mercado)
    ultima_vela_cerrada = df_mercado.iloc[-2:-1]   # Evitar "Vela Mutante"

except Exception as e:
    print(f"Error en la etapa de extracción y features: {e}")
    ultima_vela_cerrada = None

[1/4] Descargando datos y calculando indicadores...
[CCXT] Descargando BTC/USDT desde 2026-04-28...
Iniciando descarga de datos macroeconómicos...
-> Descargando SP500 (^GSPC)...
-> Descargando DXY (DX-Y.NYB)...
-> Descargando Oro (GC=F)...

¡Contexto macro agregado exitosamente!


In [8]:
ultima_vela_cerrada.T

Date,2026-06-07
Open,60884.620000
High,64234.680000
Low,60746.000000
Close,63332.010000
Volume,26612.104430
Retorno_Log,0.039410
RSI_14,24.666570
BBL_20_2.0_2.0,59793.393067
BBM_20_2.0_2.0,71758.426500
BBU_20_2.0_2.0,83723.459933
